# LIBRARY

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score 
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
#from lightgbm import LGBMClassifier
#from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from scipy.stats import randint, uniform
from collections import Counter
from collections import Counter
from imblearn.over_sampling import SMOTE
### Hyperparameter Tuning - XGBoost
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.metrics import roc_auc_score
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import randint, uniform
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform
import numpy as np



# XGBoost (SMM)

In [14]:
file_path = "../1.DATASET/CMI_FINAL_OD.csv"
df=pd.read_csv(file_path)

In [15]:
column2drop = ['id']
#column2drop = ['id', 'sii', 'Basic_Demos-Sex', 'Physical-MAP_CAL', 'FGC-FGC_CORE_CAL', 'FGC-FGC_SR_CAL', 'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_Frame_num']

df.drop(column2drop, axis=1, inplace=True)
attributes = [col for col in df.columns if col != 'BIA-BIA_SMM']
X = df[attributes].values
y = np.array(df['BIA-BIA_SMM'])

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Classes : {np.unique(y)}")
print(f"Class counts : {dict(zip(*np.unique(y, return_counts=True)))}")

X shape : (8396, 17)
y shape : (8396,)
Classes : [ 0.59056648  1.43006828  1.46180752 ... 84.3910334  89.3602
 89.4884    ]
Class counts : {0.5905664760120359: 1, 1.4300682819188886: 1, 1.461807515682601: 1, 1.6182619813301926: 1, 2.2110761126819405: 1, 2.586985953459859: 1, 2.7211293841876447: 1, 3.925588590113364: 1, 4.088467595735214: 1, 4.16082033841721: 1, 4.65573: 3, 4.690058053864007: 1, 4.906413093618454: 1, 5.113253001545111: 1, 5.138024625640213: 1, 5.147592982268819: 1, 6.082856940867913: 1, 6.174754683600405: 1, 6.216926349805693: 1, 6.249795904716159: 1, 6.347003264672487: 1, 6.501384330487781: 1, 6.704515114834571: 1, 7.625264686888286: 1, 7.685130628110869: 1, 8.416315250705217: 1, 8.42916070044232: 1, 8.7720390428405: 1, 8.800647970703755: 1, 8.836692962060372: 1, 9.055090484787955: 1, 9.239066804392795: 1, 9.322709022754648: 1, 9.43086490081118: 1, 9.609664922307282: 1, 9.751226986202926: 1, 10.474591353562298: 1, 10.571487970703757: 7, 10.769392224407746: 1, 11.027768

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

In [13]:
# 1. Train your black box (Gradient Boosting)
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=0)
gb_model.fit(X_train, y_train)

ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [ ]:
# ── Usage ─────────────────────────────────────────────────────────────────────


gb_model.fit(X_train, y_train)

# 2. Extract interpretable tree using TREPAN
tree_model, fidelity = trepan_extract(
    black_box_model = gb_model,
    X_train         = X_train,
    X_test          = X_test,
    y_test          = y_test,
    n_synthetic     = 5000,
    max_depth       = 5,
    random_state    = 0,
)

# 3. Use extracted tree directly
y_pred_tree = tree_model.predict(X_test)

In [40]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'n_estimators': [100, 500, 1000],
    'learning_rate': [0.1, 0.01, 0.001, 0.0001],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'reg_alpha': [0.1, 0.01, 0.001],
    'reg_lambda': [0.1, 0.01, 0.001],
    'max_depth': [3, 5, 7],
    'max_leaves': [0, 32, 64],
    'n_jobs': [-1],
    'min_child_weight':  randint(1, 10),   # ← importante per classi rare
    'subsample':         uniform(0.7, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
}

base_reg = XGBRegressor(
    objective='reg:squarederror',               
    tree_method='hist',             
    eval_metric='rmse',             
    random_state=42,
    n_jobs=-1,
)

rand_search = RandomizedSearchCV(
    estimator=base_reg,
    param_distributions=param_dist,
    cv=cv,
    scoring='r2', 
    n_iter=80,
    n_jobs=-1,
    refit=True,
    verbose=1,
    random_state=42,
)

In [41]:
rand_search.fit(X_train, y_train) 

Fitting 5 folds for each of 80 candidates, totalling 400 fits


/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric='rmse',
                                          feature_types=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=N...
                                        'max_leaves': [0, 32, 64],
                                        'min_child_weight': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f98622ad4f0>,
                                        'n_estimators': [100, 500, 1000],
                                        'n_jobs': [-1],
                                        'reg_alpha': [0.1, 0.01, 0.001],
                                        'reg_lambda': [0.1, 0.01, 0.001],
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f98622c0490>},
                   random_state=42, scoring='r2', verbose=1)

In [42]:
print("Best params :", rand_search.best_params_)
print(f"Best CV RMSE: {rand_search.best_score_:.4f}")

best_model = rand_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"\nRMSE (test) : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE  (test) : {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R²   (test) : {r2_score(y_test, y_pred):.4f}")

Best params : {'colsample_bytree': 0.9379501243877818, 'gamma': 0.1, 'learning_rate': 0.01, 'max_depth': 5, 'max_leaves': 64, 'min_child_weight': 9, 'n_estimators': 1000, 'n_jobs': -1, 'reg_alpha': 0.01, 'reg_lambda': 0.1, 'subsample': 0.8393346417812789}
Best CV RMSE: 0.7581

RMSE (test) : 5.9081
MAE  (test) : 4.0605
R²   (test) : 0.7552


# XGBoost (Fat)

In [43]:
file_path = "../1.DATASET/CMI_FINAL_OD.csv"
df=pd.read_csv(file_path)

In [44]:
column2drop = ['id']
#column2drop = ['id', 'sii', 'Basic_Demos-Sex', 'Physical-MAP_CAL', 'FGC-FGC_CORE_CAL', 'FGC-FGC_SR_CAL', 'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_Frame_num']

df.drop(column2drop, axis=1, inplace=True)
attributes = [col for col in df.columns if col != 'BIA-BIA_Fat']
X = df[attributes].values
y = np.array(df['BIA-BIA_Fat'])

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Classes : {np.unique(y)}")
print(f"Class counts : {dict(zip(*np.unique(y, return_counts=True)))}")

X shape : (8396, 17)
y shape : (8396,)
Classes : [2.61172319e-02 1.41335225e-01 1.55976213e-01 ... 1.26172190e+02
 1.44216233e+02 1.53820000e+02]
Class counts : {0.0261172319401445: 1, 0.1413352248065393: 1, 0.1559762127008266: 1, 0.2119670898622612: 1, 0.2683961382866897: 1, 0.2845367889101329: 1, 0.386662: 1, 0.414636: 1, 0.4217681622630778: 1, 0.447879: 1, 0.4825217355239406: 1, 0.502266828062643: 1, 0.530487: 1, 0.6921464117729261: 1, 0.6943243806978128: 1, 0.6989912268334599: 1, 0.7464369589046154: 1, 0.7695964043547789: 1, 0.7937175712902675: 1, 0.914272: 1, 0.9278624147171168: 1, 0.9361240372254578: 1, 0.960596: 1, 1.03703: 1, 1.08337: 1, 1.10211: 1, 1.12023: 1, 1.12647: 1, 1.1374242649394244: 1, 1.2181847058501525: 1, 1.23591: 1, 1.24626: 1, 1.26681: 1, 1.26814: 1, 1.37804: 1, 1.394843438253826: 1, 1.40645: 1, 1.407073571447416: 1, 1.412902699076632: 1, 1.4147: 1, 1.43457: 1, 1.43891: 1, 1.4391140395920292: 1, 1.4610040771185189: 1, 1.4630604212994474: 1, 1.46925: 1, 1.48557469

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

In [46]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [47]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'n_estimators': [100, 500, 1000],
    'learning_rate': [0.1, 0.01, 0.001, 0.0001],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'reg_alpha': [0.1, 0.01, 0.001],
    'reg_lambda': [0.1, 0.01, 0.001],
    'max_depth': [3, 5, 7],
    'max_leaves': [0, 32, 64],
    'n_jobs': [-1],
    'min_child_weight':  randint(1, 10),   # ← importante per classi rare
    'subsample':         uniform(0.7, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
}

base_reg = XGBRegressor(
    objective='reg:squarederror',               
    tree_method='hist',             
    eval_metric='rmse',             
    random_state=42,
    n_jobs=-1,
)

rand_search = RandomizedSearchCV(
    estimator=base_reg,
    param_distributions=param_dist,
    cv=cv,
    scoring='r2', 
    n_iter=80,
    n_jobs=-1,
    refit=True,
    verbose=1,
    random_state=42,
)

In [48]:
rand_search.fit(X_train, y_train) 

Fitting 5 folds for each of 80 candidates, totalling 400 fits


/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
100 fits failed out of a total of 400.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/xgboost/core.py", line 726, in inner_f
    return func(**kwargs)
  File "/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/xgboost/sklearn.py", line 1170, in fit
    self._Booster =

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric='rmse',
                                          feature_types=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=N...
                                        'max_leaves': [0, 32, 64],
                                        'min_child_weight': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f9847bf40d0>,
                                        'n_estimators': [100, 500, 1000],
                                        'n_jobs': [-1],
                                        'reg_alpha': [0.1, 0.01, 0.001],
                                        'reg_lambda': [0.1, 0.01, 0.001],
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f9847baa400>},
                   random_state=42, scoring='r2', verbose=1)

In [49]:
print("Best params :", rand_search.best_params_)
print(f"Best CV RMSE: {rand_search.best_score_:.4f}")

best_model = rand_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"\nRMSE (test) : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE  (test) : {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R²   (test) : {r2_score(y_test, y_pred):.4f}")

Best params : {'colsample_bytree': 0.9521871356061031, 'gamma': 0.1, 'learning_rate': 0.1, 'max_depth': 5, 'max_leaves': 0, 'min_child_weight': 5, 'n_estimators': 500, 'n_jobs': -1, 'reg_alpha': 0.01, 'reg_lambda': 0.01, 'subsample': 0.9157368967662602}
Best CV RMSE: 0.9867

RMSE (test) : 1.4746
MAE  (test) : 0.7652
R²   (test) : 0.9918
